# RNN e CNN para classificação Fashion-MNIST

Este notebook é uma **nova implementação comparativa**. O notebook histórico disponível em `original/` tem apenas CNN, apesar de seu nome citar RNN. Aqui cada imagem 28×28 é tratada pela RNN como **28 etapas de 28 características**. A CNN processa a imagem como estrutura espacial.

O Fashion-MNIST contém 60.000 amostras de treino e 10.000 de teste. Separamos 10% do treino para validação. O teste é utilizado somente depois do ajuste dos modelos.


## 1. Dependências e importações

Em Colab, execute as células na ordem. Instale os pacotes do `requirements.txt` se necessário.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from src.treinamento import preparar_dados, criar_modelo, formato_entrada, LABELS

tf.keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)


## 2. Dados e inspeção

In [ ]:
(x_train, y_train), (x_val, y_val), (x_test, y_test) = preparar_dados(seed=42)
print("Treino:", x_train.shape, "Validação:", x_val.shape, "Teste:", x_test.shape)
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(LABELS[y_train[i]])
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Treinamento

A comparação utiliza as mesmas partições, épocas e batch size. Diferentes arquiteturas têm capacidades e custos computacionais distintos. Resultados variam com ambiente e aleatoriedade.

In [ ]:
EPOCAS = 5
BATCH_SIZE = 128
modelos, historias, resultados = {}, {}, {}
for tipo in ("rnn", "cnn"):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)
    modelo = criar_modelo(tipo)
    modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    historia = modelo.fit(formato_entrada(x_train, tipo), y_train,
                          validation_data=(formato_entrada(x_val, tipo), y_val),
                          epochs=EPOCAS, batch_size=BATCH_SIZE, verbose=2)
    perda, acuracia = modelo.evaluate(formato_entrada(x_test, tipo), y_test, verbose=0)
    resultados[tipo] = {"perda_teste": float(perda), "acuracia_teste": float(acuracia)}
    modelos[tipo], historias[tipo] = modelo, historia
    print(tipo, resultados[tipo])


## 4. Curvas e métricas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for tipo, historia in historias.items():
    axes[0].plot(historia.history["val_accuracy"], label=tipo.upper())
    axes[1].plot(historia.history["val_loss"], label=tipo.upper())
axes[0].set_title("Acurácia de validação")
axes[1].set_title("Perda de validação")
for ax in axes:
    ax.set_xlabel("Época")
    ax.legend()
plt.tight_layout()
plt.show()

for tipo, modelo in modelos.items():
    previsoes = modelo.predict(formato_entrada(x_test, tipo), verbose=0).argmax(axis=1)
    print("\n", tipo.upper())
    print(classification_report(y_test, previsoes, target_names=LABELS, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, previsoes, display_labels=LABELS, xticks_rotation=45)
    plt.title("Matriz de confusão: " + tipo.upper())
    plt.tight_layout()
    plt.show()


## 5. Discussão

- A **RNN** percorre linhas de pixels em sequência e, embora funcione, não preserva a vizinhança espacial como uma CNN.
- A **CNN** explora padrões locais por convolução e costuma ser mais apropriada para imagens.
- Não inferir superioridade a partir de uma execução isolada. Repita com sementes diferentes e compare acurácia, F1, tempo e variabilidade.
- Valores não foram preenchidos antecipadamente: execute o notebook para produzir as métricas do ambiente utilizado.
